### Derived R0 estimation

In this notebook, we calculates the commune-level and region-level R0 summary tables.

Required input files:

- `6_metapopulation_model/outputs/inference/posterior_samples.csv`: posterior `beta_local_*` samples.
- `6_metapopulation_model/data/model_parameters.csv`: fixed `gamma` value.
- `6_metapopulation_model/data/ppr_MA_2008_region_commune_agg.xlsx`.

Outputs:
- `6_metapopulation_model/R0/r0_commune_summary.csv`
- `6_metapopulation_model/R0/r0_region_summary.csv`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in [_here, *_here.parents]
     if (path / "6_metapopulation_model" / "data").is_dir()),
    _here,
)
MODEL_DIR = REPO_ROOT / "6_metapopulation_model"
DATA_DIR = MODEL_DIR / "data"
INFERENCE_DIR = MODEL_DIR / "outputs" / "inference"
OUTPUT_DIR = MODEL_DIR / "R0"

posterior_path = INFERENCE_DIR / "posterior_samples.csv"
parameters_path = DATA_DIR / "model_parameters.csv"
outbreaks_path = DATA_DIR / "ppr_MA_2008_region_commune_agg.xlsx"
for path in (posterior_path, parameters_path, outbreaks_path):
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path}")


In [ ]:
posterior = pd.read_csv(posterior_path, encoding="utf-8-sig")
parameters = pd.read_csv(parameters_path)
outbreaks = pd.read_excel(outbreaks_path, sheet_name="Sheet1")
gamma_values = parameters.loc[
    parameters["Parameter"] == "gamma", "Value"
]
if gamma_values.empty:
    raise ValueError(f"Parameter 'gamma' not found in {parameters_path}")
gamma = float(gamma_values.iloc[0])

patches = outbreaks[["COMMUNE", "REGION"]].drop_duplicates()
patches = patches.sort_values(["REGION", "COMMUNE"]).reset_index(drop=True)

region_groups = {
    "Casablanca-Settat": "Casablanca-Settat",
    "Fés-Meknès": "Fes-Meknes",
    "Rabat-Salé-Kénitra": "Rabat-Beni",
    "Béni Mellal-Khénifra": "Rabat-Beni",
    "Daraâ-Tafilalet": "Other",
    "Oriental": "Other",
    "Marrakech-Safi": "Other",
    "Souss-Massa": "Other",
    "Tanger-Tétouan-Al Hoceima": "Other",
}
patches["class"] = patches["REGION"].map(region_groups).fillna("Other")
beta_local = {
    column.removeprefix("beta_local_"): posterior[column]
    for column in posterior.columns
    if column.startswith("beta_local_")
}
if not beta_local:
    raise ValueError("No beta_local_* columns found in posterior_samples.csv")

r0_samples = {group: values / gamma for group, values in beta_local.items()}
print(f"Posterior draws: {len(posterior)}")
print(f"gamma: {gamma}")
print(f"Infected communes: {len(patches)}")


In [ ]:
def summarize(values, total_draws):
    clean = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()
    mode_values = clean.mode()
    return {
        "n_draws": int(clean.size),
        "n_total": int(total_draws),
        "mean": float(clean.mean()),
        "median": float(clean.median()),
        "mode": float(mode_values.iloc[0]) if not mode_values.empty else np.nan,
        "sd": float(clean.std(ddof=1)) if len(clean) > 1 else np.nan,
        "q2_5": float(clean.quantile(0.025)),
        "q25": float(clean.quantile(0.25)),
        "q75": float(clean.quantile(0.75)),
        "q97_5": float(clean.quantile(0.975)),
    }

total_draws = len(posterior)
commune_rows = []
for row in patches.itertuples(index=False):
    if row[2] not in r0_samples:
        continue
    summary = summarize(r0_samples[row[2]], total_draws)
    summary.update({"COMMUNE": row[0], "REGION": row[1]})
    commune_rows.append(summary)

commune_summary = pd.DataFrame(commune_rows)[[
    "COMMUNE", "REGION", "n_draws", "n_total", "mean", "median", "mode",
    "sd", "q2_5", "q25", "q75", "q97_5",
]].sort_values(["REGION", "COMMUNE"]).reset_index(drop=True)

region_rows = []
for region, region_df in patches.groupby("REGION", sort=True):
    group = region_df["class"].iloc[0]
    if group not in r0_samples:
        continue
    summary = summarize(r0_samples[group], total_draws)
    summary.update({
        "REGION": region,
        "n_communes": int(len(region_df)),
    })
    region_rows.append(summary)

region_summary = pd.DataFrame(region_rows)[[
    "n_draws", "n_total", "mean", "median", "mode", "sd", "q2_5",
    "q25", "q75", "q97_5", "REGION", "n_communes",
]].sort_values(["median", "REGION"]).reset_index(drop=True)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
commune_output = OUTPUT_DIR / "r0_commune_summary.csv"
region_output = OUTPUT_DIR / "r0_region_summary.csv"
commune_summary.to_csv(commune_output, index=False, encoding="utf-8-sig")
region_summary.to_csv(region_output, index=False, encoding="utf-8-sig")
print(f"Saved: {commune_output}")
print(f"Saved: {region_output}")
